<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> This notebook has been tested using <strong>SageMaker Distribution Image 3.7.0</strong> and the <strong>SageMaker Python SDK version 3.4.0</strong>.
</div>

In [ ]:
!pip install -q -U "sagemaker==3.4.0"

In [ ]:
# Restart kernel to pick up the updated packages
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

# SageMaker V3 JumpStart Model Example

This notebook demonstrates how to use SageMaker V3 ModelBuilder with JumpStart models for easy model deployment and inference.

### Prerequisites
Note: Ensure you have sagemaker and ipywidgets installed in your environment. The ipywidgets package is required to monitor endpoint deployment progress in Jupyter notebooks.

In [2]:
# Import required libraries
import json
import uuid
import boto3

from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.core.jumpstart.configs import JumpStartConfig
from sagemaker.core.resources import EndpointConfig
from sagemaker.train.configs import Compute

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


## Step 1: Configure JumpStart Model

We'll use a HuggingFace Falcon model from JumpStart for this example.

In [3]:
# Configuration
# MODEL_ID = "huggingface-llm-falcon-7b-bf16"
MODEL_ID= "meta-textgeneration-llama-3-1-8b-instruct"
MODEL_NAME_PREFIX = "js-v3-example-model"
ENDPOINT_NAME_PREFIX = "js-v3-example-endpoint"

# Generate unique identifiers
unique_id = str(uuid.uuid4())[:8]
model_name = f"{MODEL_NAME_PREFIX}-{unique_id}"
endpoint_name = f"{ENDPOINT_NAME_PREFIX}-{unique_id}"

print(f"Model name: {model_name}")
print(f"Endpoint name: {endpoint_name}")

Model name: js-v3-example-model-93ff70d5
Endpoint name: js-v3-example-endpoint-93ff70d5


## Step 2: Create ModelBuilder from JumpStart Config

The ModelBuilder can automatically configure itself from a JumpStart model ID.

In [4]:
# Initialize model_builder object with JumpStart configuration
compute = Compute(instance_type="ml.g5.2xlarge")
jumpstart_config = JumpStartConfig(model_id=MODEL_ID)
model_builder = ModelBuilder.from_jumpstart_config(jumpstart_config=jumpstart_config, compute=compute)

print("ModelBuilder created successfully from JumpStart config!")

Model 'meta-textgeneration-llama-3-1-8b-instruct' requires accepting end-user license agreement (EULA). See https://jumpstart-cache-prod-us-west-2.s3.us-west-2.amazonaws.com/fmhMetadata/eula/llama3_1Eula.txt for terms of use.


[02/11/26 11:05:47] INFO     Model 'meta-textgeneration-llama-3-1-8b-instruct' requires accepting      ]8;id=842273;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/jumpstart/utils.py\utils.py]8;;\:]8;id=565642;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/jumpstart/utils.py#647\647]8;;\
                             end-user license agreement (EULA). See                                                
                             https://jumpstart-cache-prod-us-west-2.s3.us-west-2.amazonaws.com/fmhMeta             
                             data/eula/llama3_1Eula.txt for terms of use.                                          

Using model 'meta-textgeneration-llama-3-1-8b-instruct' with wildcard version identifier '*'. You can pin to version '2.38.0' for more stable results. Note that models may have different input/output signatures after a major version upgrade.


                    WARNING  Using model 'meta-textgeneration-llama-3-1-8b-instruct' with wildcard     ]8;id=74973;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/jumpstart/cache.py\cache.py]8;;\:]8;id=539274;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/jumpstart/cache.py#624\624]8;;\
                             version identifier '*'. You can pin to version '2.38.0' for more stable               
                             results. Note that models may have different input/output signatures                  
                             after a major version upgrade.                                                        

ModelBuilder created successfully from JumpStart config!


## Step 3: Build the Model

Build the model artifacts and prepare for deployment.

In [5]:
# Build the model
core_model = model_builder.build(model_name=model_name)
print(f"Model Successfully Created: {core_model.model_name}")

[02/11/26 11:05:53] INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=680603;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=7979;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#92\92]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

                    DEBUG    Either inference spec or model is provided. ModelBuilder   ]8;id=265984;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=116153;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#1315\1315]8;;\
                             is not handling MLflow model input                                                    

Model 'meta-textgeneration-llama-3-1-8b-instruct' requires accepting end-user license agreement (EULA). See https://jumpstart-cache-prod-us-west-2.s3.us-west-2.amazonaws.com/fmhMetadata/eula/llama3_1Eula.txt for terms of use.


                    INFO     Model 'meta-textgeneration-llama-3-1-8b-instruct' requires accepting      ]8;id=672997;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/jumpstart/utils.py\utils.py]8;;\:]8;id=253051;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/jumpstart/utils.py#647\647]8;;\
                             end-user license agreement (EULA). See                                                
                             https://jumpstart-cache-prod-us-west-2.s3.us-west-2.amazonaws.com/fmhMeta             
                             data/eula/llama3_1Eula.txt for terms of use.                                          

Using model 'meta-textgeneration-llama-3-1-8b-instruct' with wildcard version identifier '*'. You can pin to version '2.38.0' for more stable results. Note that models may have different input/output signatures after a major version upgrade.


...

                    DEBUG    JumpStart Model ID detected.                               ]8;id=721307;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=321822;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#2861\2861]8;;\

                    DEBUG    Building for JumpStart model ID...                               ]8;id=59300;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=598702;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py#1800\1800]8;;\

                    WARNING  Unable to check docker volume utilization at the expected path   ]8;id=170964;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/utils/local_hardware.py\local_hardware.py]8;;\:]8;id=283690;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/utils/local_hardware.py#204\204]8;;\
                             /var/lib/docker. [Errno 2] No such file or directory:                                 
                             '/var/lib/docker'                                                                     

                    INFO     Creating model with name: js-v3-example-model-93ff70d5          ]8;id=289367;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=252577;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#1814\1814]8;;\

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:2                                                                                    │
│                                                                                                  │
│   1 # Build the model                                                                            │
│ ❱ 2 core_model = model_builder.build(model_name=model_name)                                      │
│   3 print(f"Model Successfully Created: {core_model.model_name}")                                │
│   4                                                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py:168 in     │
│ wrapper                                                                                          │
│                                                                                                  │
│   165 │   │   │   │   │   caught_ex = e                                                          │
│   166 │   │   │   │   finally:                                                                   │
│   167 │   │   │   │   │   if caught_ex:                                                          │
│ ❱ 168 │   │   │   │   │   │   raise caught_ex                                                    │
│   169 │   │   │   │   │   return response  # pylint: disable=W0150                               │
│   170 │   │   │   else:                                                                          │
│   171 │   │   │   │   logger.debug(                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py:139 in     │
│ wrapper                                                                                          │
│                                                                                                  │
│   136 │   │   │   │   start_timer = perf_counter()                                               │
│   137 │   │   │   │   try:                                                                       │
│   138 │   │   │   │   │   # Call the original function                                           │
│ ❱ 139 │   │   │   │   │   response = func(*args, **kwargs)                                       │
│   140 │   │   │   │   │   stop_timer = perf_counter()                                            │
│   141 │   │   │   │   │   elapsed = stop_timer - start_timer                                     │
│   142 │   │   │   │   │   extra += f"&x-latency={round(elapsed, 2)}"                             │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:346 in       │
│ wrapper                                                                                          │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                      

## Step 4: Deploy the Model

Deploy the model to a SageMaker endpoint for real-time inference.

In [5]:
# Deploy the model to an endpoint
core_endpoint = model_builder.deploy(endpoint_name=endpoint_name)
print(f"Endpoint Successfully Created: {core_endpoint.endpoint_name}")

[02/11/26 09:37:13] INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=403277;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=127601;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#92\92]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

                    INFO     Creating endpoint-config with name                               ]8;id=696401;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=888530;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#985\985]8;;\
                             js-v3-example-endpoint-25cec61a                                                       

[02/11/26 09:37:14] INFO     Creating endpoint with name js-v3-example-endpoint-25cec61a     ]8;id=567648;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=429389;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#1017\1017]8;;\

Output()

#033[2m2026-02-11T09:41:54.527468Z#033[0m #033[32m INFO#033[0m 
#033[2mtext_generation_launcher#033[0m#033[2m:#033[0m Args { model_id: "/opt/ml/model", revision: None, 
validation_workers: 2, sharded: None, num_shard: Some(1), quantize: None, speculate: None, dtype: None, 
trust_remote_code: false, max_concurrent_requests: 128, max_best_of: 2, max_stop_sequences: 4, max_top_n_tokens: 5,
max_input_length: 1024, max_total_tokens: 2048, waiting_served_ratio: 1.2, max_batch_prefill_tokens: 4096, 
max_batch_total_tokens: None, max_waiting_tokens: 20, hostname: "container-0.local", port: 8080, shard_uds_path: 
"/tmp/text-generation-server", master_addr: "localhost", master_port: 29500, huggingface_hub_cache: Some("/tmp"), 
weights_cache_override: None, disable_custom_kernels: false, cuda_memory_fraction: 1.0, rope_scaling: None, 
rope_factor: None, json_output: false, otlp_endpoint: None, cors_allow_origin: [], watermark_gamma: None, 
watermark_delta: None, ngrok: false, ngrok_authtoken: None, ngrok_edge: None, tokenizer_config_path: None, env: 
false }

...

#033[2m2026-02-11T09:42:11.775576Z#033[0m #033[32m INFO#033[0m #033[2mtext_generation_router#033[0m#033[2m:#033[0m 
#033[2mrouter/src/main.rs#033[0m#033[2m:#033[0m#033[2m317:#033[0m Connected

✅ Created endpoint with name js-v3-example-endpoint-25cec61a

#033[2m2026-02-11T09:42:11.775581Z#033[0m #033[33m WARN#033[0m #033[2mtext_generation_router#033[0m#033[2m:#033[0m 
#033[2mrouter/src/main.rs#033[0m#033[2m:#033[0m#033[2m322:#033[0m Invalid hostname, defaulting to 0.0.0.0

[02/11/26 09:42:47] INFO     ✅ Deployment successful: Endpoint                               ]8;id=449063;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=392275;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py#1968\1968]8;;\
                             'js-v3-example-endpoint-25cec61a' using TGI in                                        
                             SAGEMAKER_ENDPOINT mode (ARN:                                                         
                             arn:aws:sagemaker:us-west-2:975049911976:endpoint/js-v3-example-                      
                             endpoint-25cec61a)                                                                    

Endpoint Successfully Created: js-v3-example-endpoint-25cec61a


## Step 5: Test the Endpoint

Send a test request to the deployed endpoint.

In [15]:
# Test the endpoint with a sample query
test_data = {"inputs": "What are falcons?", "parameters": {"max_new_tokens": 32}}

result = core_endpoint.invoke(
    body=json.dumps(test_data),
    content_type="application/json"
)

# Decode and display the result
prediction = json.loads(result.body.read().decode('utf-8'))
print(f"Model Response: {prediction}")

Model Response: [{'generated_text': '\nFalcons are birds of prey that hunt other birds and small mammals. They are found in all parts of the world except Antarctica.\nFalcons'}]


## Step 5b: Document Summarization with LangChain

Use LangChain's map-reduce chain to summarize a long document by splitting it into chunks, summarizing each chunk, then combining the summaries.


In [16]:
!pip install -q langchain langchain-aws langchain-community langchain-text-splitters

In [27]:
with open("document.txt") as f:
    text_to_summarize = f.read()
print(f"Document length: {len(text_to_summarize)} characters")

Document length: 57186 characters


In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2500, chunk_overlap=20, separators=[" "], length_function=len
)
input_documents = text_splitter.create_documents([text_to_summarize])
print(f"Split into {len(input_documents)} chunks")

Split into 24 chunks


In [34]:
from langchain_aws import SagemakerEndpoint
from langchain_aws.llms.sagemaker_endpoint import LLMContentHandler
from langchain.chains.summarize import load_summarize_chain
from langchain.prompts import PromptTemplate

class FalconContentHandler(LLMContentHandler):
    content_type = "application/json"
    accepts = "application/json"

    def transform_input(self, prompt: str, model_kwargs={}) -> bytes:
        payload = {"inputs": prompt, "parameters": {"max_new_tokens": 1024, "temperature": 0.3, "do_sample": True}}
        return json.dumps(payload).encode("utf-8")

    def transform_output(self, output: bytes) -> str:
        response_json = json.loads(output.read().decode("utf-8"))
        return response_json[0]["generated_text"]

content_handler = FalconContentHandler()

In [35]:
map_prompt = PromptTemplate(
    template="Write a concise summary of this text in a few complete sentences:\n\n{text}\n\nConcise summary:",
    input_variables=["text"]
)

combine_prompt = PromptTemplate(
    template="Combine all these summaries into a final summary in a few complete sentences:\n\n{text}\n\nFinal summary:",
    input_variables=["text"]
)

In [ ]:
summary_model = SagemakerEndpoint(
    endpoint_name=core_endpoint.endpoint_name,
    region_name=boto3.Session().region_name,
    content_handler=content_handler,
)

summary_chain = load_summarize_chain(
    llm=summary_model,
    chain_type="map_reduce",
    map_prompt=map_prompt,
    combine_prompt=combine_prompt,
    verbose=True,
)

import boto3
result = summary_chain.invoke({"input_documents": input_documents, "token_max": 2048})
print("\n=== Final Summary ===")
print(result["output_text"])



> Entering new MapReduceDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Write a concise summary of this text in a few complete sentences:

Review
A systematic review of automatic text summarization for
biomedical literature and EHRs
Mengqian Wang ,1 Manhua Wang,2 Fei Yu ,2,3Yue Yang,1Jennifer Walker,3 and
Javed Mostafa1,2,4
1Carolina Health Informatics Program, University of North Carolina, Chapel Hill, North Carolina, USA, 2iSchool, University of North
Carolina, Chapel Hill, North Carolina, USA, 3Health Sciences Library, University of North Carolina, Chapel Hill, North Carolina,
USA, and 4Biomedical Research Imaging Center, the School of Medicine, University of North Carolina, Chapel Hill, North Carolina,
USA
*Corresponding Author: Mengqian Wang, MA, Carolina Health Informatics Program, 335 S Columbia St., CB#7585, Chapel
Hill, NC 27599-7585, USA (mengqian@email.unc.edu )
Received 6 February 2021; Revised 21 June 2021; Editorial Decision 22 June 2

## Step 6: Clean Up Resources

Clean up the created resources to avoid ongoing charges.

In [7]:
# Clean up resources
core_endpoint_config = EndpointConfig.get(endpoint_config_name=core_endpoint.endpoint_name)

# Delete in the correct order
core_model.delete()
core_endpoint.delete()
core_endpoint_config.delete()

print("All resources successfully deleted!")

[02/11/26 09:45:24] INFO     Deleting Model - js-v3-example-model-25cec61a                       ]8;id=108717;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=46034;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#22918\22918]8;;\

                    INFO     Deleting Endpoint - js-v3-example-endpoint-25cec61a                 ]8;id=565834;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=42719;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#10939\10939]8;;\

                    INFO     Deleting EndpointConfig - js-v3-example-endpoint-25cec61a           ]8;id=153891;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=34622;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#11753\11753]8;;\

All resources successfully deleted!


## Summary

This notebook demonstrated:
1. Creating a ModelBuilder from JumpStart configuration
2. Building a model from JumpStart
3. Deploying to a SageMaker endpoint
4. Making inference requests
5. Cleaning up resources

The V3 ModelBuilder makes it easy to work with JumpStart models with minimal configuration required!